In [1]:

# STEP 1: INSPECT THE INDOFLOODS FLOOD-EVENT DATASET

import pandas as pd
from pathlib import Path

# Path to the raw INDOFLOODS data folder.
raw_folder = Path("../../Raw_Data/INDOFLOOD_Data")

# Locate the flood-event CSV.
matches = list(raw_folder.glob("floodevents_indofloods.csv"))

if not matches:
    print("File not found. CSV files in the folder:")
    for file_path in sorted(raw_folder.glob("*.csv")):
        print(file_path.name)
else:
    # Load the dataset without modifying the original file.
    df_flood_events = pd.read_csv(matches[0], low_memory=False)

    # Inspect its size, columns, types, and missing values.
    print("Dataset shape:", df_flood_events.shape)
    print("\nColumns:")
    print(df_flood_events.columns.tolist())

    print("\nData types:")
    print(df_flood_events.dtypes)

    print("\nMissing values:")
    print(df_flood_events.isna().sum().sort_values(ascending=False))

    print("\nFirst five rows:")
    print(df_flood_events.head().to_string())


Dataset shape: (4548, 13)

Columns:
['EventID', 'Start Date', 'End Date', 'Peak Flood Level (m)', 'Peak FL Date', 'Num Peak FL', 'Peak Discharge Q (cumec)', 'Peak Discharge Date', 'Flood Volume (cumec)', 'Event Duration (days)', 'Time to Peak (days)', 'Recession Time (day)', 'Flood Type']

Data types:
EventID                         str
Start Date                      str
End Date                        str
Peak Flood Level (m)        float64
Peak FL Date                    str
Num Peak FL                   int64
Peak Discharge Q (cumec)    float64
Peak Discharge Date             str
Flood Volume (cumec)        float64
Event Duration (days)         int64
Time to Peak (days)           int64
Recession Time (day)          int64
Flood Type                      str
dtype: object

Missing values:
Flood Volume (cumec)        577
Peak Discharge Q (cumec)    555
Peak Discharge Date         555
Start Date                    0
EventID                       0
Peak FL Date                  0
Peak F

In [2]:

# STEP 2: COMPLETE PREPROCESSING OF INDOFLOODS FLOOD EVENTS

import pandas as pd
from pathlib import Path

# 1. Define the raw input and processed output paths.
raw_path = Path("../../Raw_Data/INDOFLOOD_Data/floodevents_indofloods.csv")
output_path = Path("floodevents_indofloods_processed.csv")

# 2. Load the original dataset.
df_flood_events = pd.read_csv(raw_path, low_memory=False)

# 3. Create a separate working copy to protect the raw data.
df_flood_processed = df_flood_events.copy()

# 4. Clean column names by removing extra spaces.
df_flood_processed.columns = (
    df_flood_processed.columns.astype(str).str.strip()
)

# 5. Remove columns that are completely empty.
empty_columns = df_flood_processed.columns[
    df_flood_processed.isna().all()
].tolist()

df_flood_processed = df_flood_processed.drop(
    columns=empty_columns
)

# 6. Convert event dates to datetime.
date_columns = [
    "Start Date",
    "End Date",
    "Peak FL Date",
    "Peak Discharge Date"
]

for column in date_columns:
    if column in df_flood_processed.columns:
        df_flood_processed[column] = pd.to_datetime(
            df_flood_processed[column],
            errors="coerce"
        )

# 7. Convert measurement and duration columns to numeric values.
numeric_columns = [
    "Peak Flood Level (m)",
    "Num Peak FL",
    "Peak Discharge Q (cumec)",
    "Flood Volume (cumec)",
    "Event Duration (days)",
    "Time to Peak (days)",
    "Recession Time (day)"
]

for column in numeric_columns:
    if column in df_flood_processed.columns:
        df_flood_processed[column] = pd.to_numeric(
            df_flood_processed[column],
            errors="coerce"
        )

# 8. Trim extra spaces in text fields.
for column in df_flood_processed.select_dtypes(
    include=["object", "string"]
).columns:
    df_flood_processed[column] = (
        df_flood_processed[column].map(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )
    )

# 9. Remove exact duplicate rows.
duplicates_removed = int(
    df_flood_processed.duplicated().sum()
)

df_flood_processed = (
    df_flood_processed.drop_duplicates().copy()
)

# 10. Display preprocessing results.
print("Original dataset shape:", df_flood_events.shape)
print("Empty columns removed:", empty_columns)
print("Duplicate rows removed:", duplicates_removed)
print("Processed dataset shape:", df_flood_processed.shape)

print("\nData types after processing:")
print(df_flood_processed.dtypes)

print("\nMissing values after processing:")
print(
    df_flood_processed.isna()
    .sum()
    .sort_values(ascending=False)
)

# 11. Save the processed dataset.
df_flood_processed.to_csv(output_path, index=False)

print("\nProcessed dataset saved to:")
print(output_path.resolve())

# 12. Save a preprocessing report.
report_path = Path("floodevents_indofloods_preprocessing_report.txt")

with open(report_path, "w", encoding="utf-8") as report:
    report.write("INDOFLOODS FLOOD EVENTS PREPROCESSING REPORT\n")
    report.write(f"Original shape: {df_flood_events.shape}\n")
    report.write(f"Empty columns removed: {empty_columns}\n")
    report.write(f"Duplicate rows removed: {duplicates_removed}\n")
    report.write(f"Processed shape: {df_flood_processed.shape}\n")
    report.write("\nMissing values after processing:\n")
    report.write(
        df_flood_processed.isna()
        .sum()
        .sort_values(ascending=False)
        .to_string()
    )

print("Preprocessing report saved to:")
print(report_path.resolve())


Original dataset shape: (4548, 13)
Empty columns removed: []
Duplicate rows removed: 0
Processed dataset shape: (4548, 13)

Data types after processing:
EventID                                str
Start Date                  datetime64[us]
End Date                    datetime64[us]
Peak Flood Level (m)               float64
Peak FL Date                datetime64[us]
Num Peak FL                          int64
Peak Discharge Q (cumec)           float64
Peak Discharge Date         datetime64[us]
Flood Volume (cumec)               float64
Event Duration (days)                int64
Time to Peak (days)                  int64
Recession Time (day)                 int64
Flood Type                             str
dtype: object

Missing values after processing:
Flood Volume (cumec)        577
Peak Discharge Q (cumec)    555
Peak Discharge Date         555
Start Date                    0
EventID                       0
Peak FL Date                  0
Peak Flood Level (m)          0
End Date        